In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from os.path import getsize

**Parsing functions**

In [ ]:
def get_next_analysed_game():
    # Until we find a game with evals
    while True:
        # Record game info
        buffer = []
        while True:
            line = f.readline()

            if line == "\n":
                continue

            buffer.append(line)

            # Moves always start with 1.
            if line.startswith("1."):
                break
                
        # We need games with move evaluations
        if ("%eval" in line):
            return buffer

In [ ]:
def params_to_dict(str_list):

    return {
        a: b.strip('"') 
        for a, b in [
            i.strip("\n").strip("[]").split(" ", 1) 
            for i in str_list
        ]
    }

In [ ]:
def moves_to_df(moves):
    s = moves.replace("[", "").replace("]", "")
    s = s.split(" ")
    s = s[:-1]
    
    # Small fix for when last move is mate
    if len(s) % 8 != 0:
        s.insert(-3, "%eval")
        s.insert(-3, "#0")

    df = pd.DataFrame.from_dict({
        "Move": s[1::8],
        "Eval": s[4::8],
        "Clock": s[6::8]
    }, orient="index").transpose()
    
    df["MoveNumber"] = (df.index // 2) + 1
    df["MoveSide"] = (df.index % 2)
    
    # Only first 200 moves have analysis
    df = df.head(200)

    return df

In [ ]:
def get_and_parse_next_analysed_game():
    while True:
        data = get_next_analysed_game()
        game = params_to_dict(data[:-1])
        
        is_good_game = all([
            game["TimeControl"].split("+")[0] in ["600", "900"],
            abs(float(game["WhiteRatingDiff"])) <= 20,
            abs(float(game["BlackRatingDiff"])) <= 20,
            abs(int(game["WhiteElo"]) - int(game["BlackElo"])) <= 100,
            game["Termination"] in ["Normal", "Time forfeit"]
        ])
        
        if not is_good_game:
            continue
        
        moves = moves_to_df(data[-1])
        
        FirstMoves = moves["Move"].values[:20]
        FirstMoves = " ".join(FirstMoves)
        FirstMoves = FirstMoves.replace("?", "").replace("!", "").replace("+", "").replace("#", "")
        game["FirstMoves"] = FirstMoves

        return (
            game, 
            moves
        )

In [ ]:
def get_two_dfs(size=10_000):
    games_list = []
    moves_list = []
    n_good_games_found = 0
    while n_good_games_found < size:
        
        try:
            a, b = get_and_parse_next_analysed_game()
        except:
            continue

        game_id = a["Site"].split("/")[-1]
        a["GameId"] = game_id
        b["GameId"] = game_id
        
        games_list.append(a)
        moves_list.append(b)
        
        n_good_games_found += 1
        
    return (
        pd.DataFrame(games_list),
        pd.concat(moves_list)
    )

**Start parsing**

In [ ]:
PGN_FILE = "pgn/lichess_db_standard_rated_2024-01.pgn"
print(f"PGN file size (bytes): {getsize(PGN_FILE):,}")

f = open(PGN_FILE, mode="r")

# Start parsing from a centain point

# OFFSET = 0
# f.seek(OFFSET)
# while True:
#     line = f.readline()
#     if line.startswith("1."):
#         break

In [ ]:
batch_size = 10_000
n_batches = 150

In [ ]:
for batch in range(1, n_batches + 1):

    df_games, df_moves = get_two_dfs(batch_size)

    df_games = df_games[[
        "GameId",
        "Result",
        "TimeControl",
        "Termination",
        
        "White",
        "Black",
        "WhiteElo",
        "BlackElo",
        "WhiteRatingDiff",
        "BlackRatingDiff",
        "ECO",
        "Opening",
        "FirstMoves"
    ]]

    df_games["WhiteElo"] = df_games["WhiteElo"].astype(int)
    df_games["BlackElo"] = df_games["BlackElo"].astype(int)

    df_games["WhiteRatingDiff"] = df_games["WhiteRatingDiff"].astype(int)
    df_games["BlackRatingDiff"] = df_games["BlackRatingDiff"].astype(int)
    
    # Assert ids are the same in both dfs
    ids = set(df_games["GameId"]) & set(df_moves["GameId"])
    df_games = df_games[ df_games["GameId"].isin(ids) ]
    df_moves = df_moves[ df_moves["GameId"].isin(ids) ]
    # assert set(df_games["GameId"]) == set(df_moves["GameId"])

    df_games.to_parquet(f"01_parsed/batch_{batch}_games.parquet")
    df_moves.to_parquet(f"01_parsed/batch_{batch}_moves.parquet")

    print(f"Saved batch #{batch}", end="\r")

In [ ]:
print(f"Last cursor position (bytes): {f.tell():,}")

In [ ]:
# Last cursor position (bytes):